In [16]:
# ==========================================
# CELL 1: SETUP API & PATH FILE (VERSI GEMINI)
# ==========================================
import os
import time
import pandas as pd
import google.generativeai as genai

# Setup API Key Gemini
# Ganti dengan key asli Anda jika os.getenv gagal: api_key_gemini = "AIzaSy..."
api_key_gemini = os.getenv("GEMINI_API_KEY")

if api_key_gemini:
    # Cek apakah terbaca (Print 8 karakter pertama saja untuk keamanan)
    print(f"🔑 Key yang terbaca oleh sistem: {api_key_gemini[:8]}...") 
    # Jika hasil print memunculkan 'AIza...' (tanpa tanda kutip), berarti SUKSES!
    
    # Konfigurasi Gemini API
    genai.configure(api_key=api_key_gemini)
    
    # Inisialisasi Model (Disiapkan untuk Cell 3 nanti)
    model = genai.GenerativeModel('gemini-2.5-flash')
    print("✅ Client Gemini 2.5 Flash berhasil diinisialisasi!")
else:
    print("❌ ERROR: API Key Gemini tidak ditemukan! Pastikan environment variable sudah diset.")

# Setup Path (Sesuaikan dengan struktur folder repo exigen-smart-maintenance Anda)
path_df_aset = "../../data/master_aset_enriched.xlsx" 
path_df_perbaikan = "../../data/aset_komplain_enriched.xlsx"
path_output = "../../data/dataset_keluhan_ntg.csv"

print("✅ Cell 1 Selesai: Library ter-import dan Path sudah diset.")

🔑 Key yang terbaca oleh sistem: AIzaSyBh...
✅ Client Gemini 2.5 Flash berhasil diinisialisasi!
✅ Cell 1 Selesai: Library ter-import dan Path sudah diset.


In [9]:
# ==========================================
# CELL 2: LOAD DATA & AMBIL ATURAN (CONSTRAINTS)
# ==========================================
print("Membaca dataset asli NTG...")
df_aset = pd.read_excel(path_df_aset)
df_perbaikan = pd.read_excel(path_df_perbaikan)

# Mengambil daftar Kunci Jawaban (Target Y) agar AI tidak ngarang
list_severity = df_perbaikan['Severity'].dropna().unique().tolist()
list_penyebab = df_perbaikan['Penyebab'].dropna().unique().tolist()
list_jenis_kerusakan = df_perbaikan['Jenis Kerusakan'].dropna().unique().tolist() 

# Mengambil Kombinasi Aset Nyata (Feature X)
kombinasi_valid = df_aset[['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 
                           'Lokasi Gedung', 'Lokasi Lantai', 'Lokasi Zona']].dropna().drop_duplicates()

# UNTUK TESTING: Kita ambil 3 kombinasi acak saja dulu agar prosesnya cuma hitungan detik
kombinasi_sample = kombinasi_valid.sample(3) 

print(f"✅ Cell 2 Selesai: Berhasil mengekstrak {len(kombinasi_sample)} kombinasi mesin untuk di-testing.")
print(f"🔸 Batas Severity yang diizinkan: {list_severity[:3]}...")

Membaca dataset asli NTG...
✅ Cell 2 Selesai: Berhasil mengekstrak 3 kombinasi mesin untuk di-testing.
🔸 Batas Severity yang diizinkan: ['Fatal', 'Ringan', 'Sedang']...


In [10]:
# ==========================================
# CELL 2.1: VALIDASI DATA UNIK (MAKESURE DATA BENAR)
# ==========================================

print("🔍 === RINGKASAN DATA UNIK UNTUK PROMPT AI === 🔍\n")

# 1. Validasi Kolom Target (dari df_perbaikan)
print("--- [TARGET Y / KUNCI JAWABAN] ---")
print(f"🔹 Severity ({len(list_severity)}): {list_severity}")
print(f"🔹 Penyebab ({len(list_penyebab)}): {list_penyebab[:10]}... (dan seterusnya)")
print(f"🔹 Tindakan ({len(list_jenis_kerusakan)}): {list_jenis_kerusakan[:10]}... (dan seterusnya)")

# 2. Validasi Identitas & Lokasi (dari df_aset)
print("\n--- [IDENTITAS ASET & LOKASI] ---")
cols_aset = ['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 'Lokasi Gedung', 'Lokasi Lantai', 'Lokasi Zona']

for col in cols_aset:
    unique_vals = df_aset[col].dropna().unique().tolist()
    print(f"📍 {col} ({len(unique_vals)} unik): {unique_vals[:15]}") # Tampilkan 15 contoh pertama

# 3. Validasi Kombinasi yang Terpilih untuk Testing
print("\n--- [CONTOH KOMBINASI UNTUK TESTING GROQ] ---")
display(kombinasi_sample)

🔍 === RINGKASAN DATA UNIK UNTUK PROMPT AI === 🔍

--- [TARGET Y / KUNCI JAWABAN] ---
🔹 Severity (4): ['Fatal', 'Ringan', 'Sedang', 'Berat']
🔹 Penyebab (12): ['Kelembaban tinggi', 'Human error', 'Overload', 'Kurang perawatan', 'Tegangan tidak stabil', 'Faktor lingkungan', 'Debu/kotoran', 'Aus normal', 'Usia pakai', 'Kualitas material']... (dan seterusnya)
🔹 Tindakan (20): ['Retak/pecah', 'Overheat', 'Remote tidak berfungsi', 'Getaran berlebihan', 'Korsleting', 'Aus/abrasi', 'Tidak berfungsi total', 'Kebocoran', 'Sensor error', 'Aliran lemah']... (dan seterusnya)

--- [IDENTITAS ASET & LOKASI] ---
📍 Kategori (15 unik): ['Mechanical', 'Ventilasi Sistem', 'Electrical', 'Sistem Pemadam Kebakaran', 'Sistem Telekomunikasi Gedung', 'Sistem Proteksi Kebakaran Aktif', 'Security Sistem', 'Civil', 'Plumbing', 'Distribusi Air', 'Sistem Transportasi Gedung', 'Pencatatan Meter', 'Arsitektur', 'Sistem Energi', 'Latihan Balakar']
📍 Sub Kategori (51 unik): ['Tata Udara', 'Sistem Sirkulasi Udara', 'Contro

,Kategori,Sub Kategori,Tipe,Merek,Lokasi Gedung,Lokasi Lantai,Lokasi Zona
9254,Mechanical,Tata Udara,AC Split,Daikin,Gedung A,3,Tengah
43332,Plumbing,Sistem Pemipaan gedung,Pemipaan Air Bersih,Lokal,Gedung Utama,2,Selatan
13220,Security Sistem,Sistem Pengawasan,DVR CCTV,Dahua,Gedung D,10,Selatan


In [11]:
# ==========================================
# CELL 2.2: STRATIFIED SAMPLING (JAMINAN 100% COVERAGE)
# ==========================================

print("Membuat antrean kasus komprehensif agar tidak ada variasi yang terlewat...")

# 1. Ambil SEMUA variasi unik dari histori perbaikan (Dijamin 100% terwakili)
kasus_unik = df_perbaikan.drop_duplicates(
    subset=['Kategori', 'Severity', 'Penyebab', 'Jenis Kerusakan']
).copy()

print(f"Total variasi kasus unik di histori: {len(kasus_unik)} kombinasi.")

# 2. Pasangkan setiap kasus unik dengan spesifikasi Aset Nyata (Merek, Lokasi, dll)
antrean_prompt = []

for _, row in kasus_unik.iterrows():
    kategori_kasus = row['Kategori']
    
    # Cari aset di df_aset yang Kategori-nya cocok dengan kasus ini
    aset_cocok = kombinasi_valid[kombinasi_valid['Kategori'] == kategori_kasus]
    
    if not aset_cocok.empty:
        # Ambil 1 aset acak yang relevan untuk dijadikan "aktor" dalam skenario ini
        aset_pilih = aset_cocok.sample(1).iloc[0]
        
        antrean_prompt.append({
            'Kategori': kategori_kasus,
            'Tipe': aset_pilih['Tipe'],
            'Merek': aset_pilih['Merek'],
            'Lokasi Gedung': aset_pilih['Lokasi Gedung'],
            'Lokasi Lantai': aset_pilih['Lokasi Lantai'],
            'Lokasi Zona': aset_pilih['Lokasi Zona'],
            'Severity': row['Severity'],
            'Penyebab': row['Penyebab'],
            'Jenis Kerusakan': row['Jenis Kerusakan'],
            'Biaya Perbaikan': row['Biaya Perbaikan']
        })

df_antrean = pd.DataFrame(antrean_prompt)
print(f"✅ Antrean Siap! Total data yang HARUS di-generate AI: {len(df_antrean)} baris.")
display(df_antrean.head())

Membuat antrean kasus komprehensif agar tidak ada variasi yang terlewat...
Total variasi kasus unik di histori: 9000 kombinasi.
✅ Antrean Siap! Total data yang HARUS di-generate AI: 9000 baris.


,Kategori,Tipe,Merek,Lokasi Gedung,Lokasi Lantai,Lokasi Zona,Severity,Penyebab,Jenis Kerusakan,Biaya Perbaikan
0,Mechanical,AC Cassette,Mitsubishi,Gedung C,5,Selatan,Fatal,Kelembaban tinggi,Retak/pecah,18771000
1,Security Sistem,Kamera CCTV,Axis,Gedung A,8,Timur,Ringan,Human error,Overheat,356000
2,Electrical,Sistem Grounding,Generic,Gedung B,4,Timur,Sedang,Overload,Remote tidak berfungsi,1610000
3,Mechanical,FCU,Generic,Gedung Utama,11,Utara,Ringan,Kurang perawatan,Overheat,122000
4,Security Sistem,Kamera CCTV,Axis,Gedung C,2,Barat,Sedang,Tegangan tidak stabil,Getaran berlebihan,1760000


In [17]:
# ==========================================
# CELL 3: GENERATE DATASET VIA GEMINI (RESUME MODE)
# ==========================================
import time
import os
import pandas as pd
import google.generativeai as genai

path_output = "../../data/dataset_keluhan_ntg.csv" # Sesuaikan path Anda

# 1. BACA FILE LAMA UNTUK CEK YANG SUDAH SELESAI
kombinasi_selesai = set()
if os.path.exists(path_output):
    try:
        df_lama = pd.read_csv(path_output, sep='|', on_bad_lines='skip')
        for _, row in df_lama.iterrows():
            kunci = f"{row['kategori_aset']}_{row['severity']}_{row['root_cause']}_{row['jenis_kerusakan']}"
            kombinasi_selesai.add(kunci)
        print(f"✅ Menemukan file lama. {len(df_lama)} riwayat kasus dipulihkan.")
    except Exception as e:
        print(f"⚠️ Gagal membaca file lama, mulai dari awal. Error: {e}")

# 2. FILTER ANTREAN (Hanya ambil yang belum ada di file CSV)
antrean_sisa = []
for index, row in df_antrean.iterrows():
    kunci_antrean = f"{row['Kategori']}_{row['Severity']}_{row['Penyebab']}_{row['Jenis Kerusakan']}"
    if kunci_antrean not in kombinasi_selesai:
        antrean_sisa.append((index, row))

print(f"🎯 Total variasi unik keseluruhan: {len(df_antrean)}")
print(f"⏩ Sisa yang belum dikerjakan AI: {len(antrean_sisa)}\n")

# 3. PROSES LOOPING API GEMINI
csv_batch_baru = "" # Variabel penampung hasil baru

if len(antrean_sisa) > 0:
    print(f"🚀 Mulai memanggil Gemini 1.5 Flash untuk sisa antrean...")
    for index_asli, row in antrean_sisa:
        kat, tipe, merek = row['Kategori'], row['Tipe'], row['Merek']
        ged, lan, zon = row['Lokasi Gedung'], row['Lokasi Lantai'], row['Lokasi Zona']
        sev, rc, jk, biaya = row['Severity'], row['Penyebab'], row['Jenis Kerusakan'], row['Biaya Perbaikan']
        
        print(f"🔄 Memproses [Antrean {index_asli+1}/{len(df_antrean)}]: {kat} - {jk}...")
        
        # PROMPT GEMINI (System instruction digabung ke dalam teks prompt)
        prompt = f"""
        Anda adalah AI pembuat dataset Machine Learning. TUGAS ANDA HANYA MENGELUARKAN TEKS CSV MURNI TANPA HEADER. JANGAN ADA TEKS BASA-BASI.
        
        Buatkan 3 baris data CSV (3 variasi kalimat berbeda) untuk SATU skenario spesifik ini:
        - Kategori: {kat} (Tipe: {tipe}, Merek: {merek})
        - Lokasi: {ged} Lantai {lan} Zona {zon}
        - Severity: {sev}
        - Root Cause: {rc}
        - Jenis Kerusakan: {jk}
        - Biaya Perbaikan: {biaya}

        Format WAJIB: CSV murni dengan pemisah '|'.
        Kolom: teks_keluhan_awam|teks_laporan_teknisi|kategori_aset|severity|root_cause|jenis_kerusakan|biaya_perbaikan

        ATURAN KETAT:
        1. Anda WAJIB menggunakan nilai Severity, Root Cause, Jenis Kerusakan, Kategori, dan Biaya Perbaikan PERSIS seperti skenario di atas.
        2. Tugas Anda hanya mengarang 3 variasi berbeda dari 'teks_keluhan_awam' dan 'teks_laporan_teknisi'. Selipkan merek dan lokasi secara natural.
        3. OUTPUT MURNI CSV. JANGAN ada markdown (```csv). JANGAN keluarkan header.
        """
        
        try:
            # 🌟 SINTAKS GEMINI YANG BENAR 🌟
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.7,
                    max_output_tokens=2000,
                )
            )
            
            # 🌟 PARSING RESPONSE GEMINI 🌟
            hasil_text = response.text.strip() 
            
            # Bersihkan markdown jika masih ada
            hasil_text = hasil_text.replace("```csv", "").replace("```", "").strip()
            
            # Hapus header jika AI tidak sengaja mengeluarkannya
            if hasil_text.lower().startswith("teks_keluhan"):
                hasil_text = "\n".join(hasil_text.split("\n")[1:])
                
            csv_batch_baru += hasil_text + "\n" # Tampung di memori
            
            # Jeda 5 detik agar aman dari batas 15 Request/Menit Gemini gratisan
            time.sleep(5) 
            
        except Exception as e:
            print(f"❌ Terkena Limit/Error pada baris {index_asli+1}. Looping dihentikan aman. Error: {e}")
            break # Berhenti, tapi csv_batch_baru TIDAK HILANG

    print("✅ Cell 3 Selesai! Silakan jalankan Cell 4 untuk menyimpan data yang baru di-generate.")
else:
    print("🎉 Hore! Semua variasi kasus sudah selesai di-generate. Tidak perlu memanggil API lagi.")

✅ Menemukan file lama. 367 riwayat kasus dipulihkan.
🎯 Total variasi unik keseluruhan: 9000
⏩ Sisa yang belum dikerjakan AI: 8884

🚀 Mulai memanggil Gemini 1.5 Flash untuk sisa antrean...
🔄 Memproses [Antrean 59/9000]: Electrical - Performa menurun...
🔄 Memproses [Antrean 72/9000]: Mechanical - Korsleting...
🔄 Memproses [Antrean 94/9000]: Arsitektur - Korsleting...
🔄 Memproses [Antrean 110/9000]: Civil - Korsleting...
🔄 Memproses [Antrean 112/9000]: Arsitektur - Suhu tidak stabil...
🔄 Memproses [Antrean 118/9000]: Arsitektur - Kebocoran...
🔄 Memproses [Antrean 123/9000]: Civil - Bunyi abnormal...
🔄 Memproses [Antrean 124/9000]: Mechanical - Aus/abrasi...
🔄 Memproses [Antrean 125/9000]: Mechanical - Tampilan rusak...
🔄 Memproses [Antrean 126/9000]: Mechanical - Bunyi abnormal...
🔄 Memproses [Antrean 127/9000]: Mechanical - Mati mendadak...
🔄 Memproses [Antrean 128/9000]: Arsitektur - Korsleting...
🔄 Memproses [Antrean 129/9000]: Sistem Telekomunikasi Gedung - Lampu mati...
🔄 Memproses [

In [18]:
# ==========================================
# CELL 4: SIMPAN & BACA HASIL (APPEND MODE)
# ==========================================

# 1. Simpan Data Baru (Jika ada)
if 'csv_batch_baru' in locals() and csv_batch_baru.strip() != "":
    
    # Jika file belum pernah ada sama sekali, buatkan sekalian beserta headernya
    if not os.path.exists(path_output):
        with open(path_output, "w", encoding="utf-8") as f:
            f.write("teks_keluhan_awam|teks_laporan_teknisi|kategori_aset|severity|root_cause|jenis_kerusakan|biaya_perbaikan\n")
            
    # Tambahkan (Append) data baru ke baris paling bawah file CSV
    with open(path_output, "a", encoding="utf-8") as f:
        f.write(csv_batch_baru)
        
    print(f"💾 File berhasil di-update dan disimpan di: {path_output}")
    
    # KOSONGKAN variabel setelah disimpan agar tidak ter-save ganda jika Anda me-run Cell 4 dua kali
    csv_batch_baru = "" 
else:
    print("ℹ️ Tidak ada teks baru untuk ditambahkan ke file saat ini.")

# 2. Coba baca keseluruhan dataset menggunakan Pandas
try:
    df_hasil = pd.read_csv(path_output, sep='|', on_bad_lines='skip')
    print(f"\n📊 DATASET FINAL! Total Keseluruhan Data Saat Ini: {df_hasil.shape[0]} baris.")
    display(df_hasil.head())
    display(df_hasil.tail()) # Tampilkan juga bagian paling bawah untuk ngecek
except Exception as e:
    print(f"❌ Gagal membaca CSV. Error: {e}")

💾 File berhasil di-update dan disimpan di: ../../data/dataset_keluhan_ntg.csv

📊 DATASET FINAL! Total Keseluruhan Data Saat Ini: 415 baris.


,teks_keluhan_awam,teks_laporan_teknisi,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
0,AC Split Samsung di Gedung Utama Lantai 11 Zon...,Laporan teknisi: AC Split Samsung di Gedung Ut...,Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000
1,AC Split Samsung di kantor Gedung Utama Lantai...,"Laporan teknisi: Setelah inspeksi, ditemukan b...",Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000
2,Saya menemukan AC Split Samsung di Gedung Utam...,Laporan teknisi: AC Split Samsung di Gedung Ut...,Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000
3,Sistem keamanan di Gedung B Lantai 5 Zona Teng...,Laporan teknisi: Sistem access control merek l...,Security Sistem,Ringan,Human error,Overheat,356000
4,Sistem akses kontrol merek lokal di Gedung B L...,Laporan teknisi: Sistem access control merek l...,Security Sistem,Ringan,Human error,Overheat,356000


,teks_keluhan_awam,teks_laporan_teknisi,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
410,Tolong dicek AC di area kerja Gedung B Lantai ...,Unit FCU Generic di Gedung B Lantai 10 Zona Ut...,Mechanical - FCU - Generic,Sedang,Kurang perawatan,Suhu tidak stabil,1249000
411,AC di bagian utara Lantai 10 Gedung B terasa a...,Ditemukan masalah suhu tidak stabil pada FCU G...,Mechanical - FCU - Generic,Sedang,Kurang perawatan,Suhu tidak stabil,1249000
412,Ada bercak dan kusam di lantai parkir Gedung P...,Ditemukan kerusakan tampilan pada permukaan la...,"Arsitektur (Lantai Parkir Beton, Merek: Import)",Sedang,Korosi,Tampilan rusak,1544000
413,Lantai parkir di Gedung Parkir Lantai 1 Zona U...,Evaluasi menunjukkan adanya kerusakan tampilan...,"Arsitektur (Lantai Parkir Beton, Merek: Import)",Sedang,Korosi,Tampilan rusak,1544000
414,Saya lihat lantai parkir di Gedung Parkir Lant...,Laporan teknisi mengkonfirmasi kerusakan tampi...,"Arsitektur (Lantai Parkir Beton, Merek: Import)",Sedang,Korosi,Tampilan rusak,1544000
